# 8 — Block 1 relation splitting policy

Purpose: make the Block 1 split/no-split policy executable and auditable for `gene_interacts_gene`, `pathway_contains_gene`, and `molecule_targets_gene`.

Core doctrine: enumerate source/subdatabase detail in evidence first, then split only when source assertion and endpoint namespace justify a narrower relation. Do not project gene-level rows to protein just because a gene→protein mapping exists.

In [ ]:
from __future__ import annotations
import json, os, sys
from dataclasses import asdict, is_dataclass
from pathlib import Path
import duckdb
import pandas as pd
import pyarrow.parquet as pq
try:
    from IPython import get_ipython
    if get_ipython() is None:
        raise RuntimeError("not running inside IPython")
    from IPython.display import display
except Exception:  # pragma: no cover - plain Python fallback for lightweight validation
    def display(obj):
        print(obj)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "manage_db").exists():
    REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

SAMPLE_MODE = os.environ.get("JOUVENCE_NOTEBOOK_SAMPLE_MODE", os.environ.get("TXGNN_NOTEBOOK_SAMPLE_MODE", "1")) != "0"
RUN_BUILD = os.environ.get("JOUVENCE_NOTEBOOK_RUN_BUILD", os.environ.get("TXGNN_NOTEBOOK_RUN_BUILD", "0")) == "1"
RUN_FULL_VALIDATION = os.environ.get("JOUVENCE_NOTEBOOK_FULL_VALIDATION", os.environ.get("TXGNN_NOTEBOOK_FULL_VALIDATION", "0")) == "1"
FUSE_KG_ROOT = Path("/Users/jkobject/mnt/gcs/jouvencekb-kg/v2")
def path_exists(path: Path) -> bool:
    try:
        return path.exists()
    except OSError:
        return False
DEFAULT_KG_ROOT = Path(os.environ.get("JOUVENCE_KG_ROOT", os.environ.get("TXGNN_KG_ROOT", str(FUSE_KG_ROOT if path_exists(FUSE_KG_ROOT) else REPO_ROOT / "data" / "kg"))))
DATA_DIR = Path(os.environ.get("JOUVENCE_DATA_DIR", os.environ.get("TXGNN_DATA_DIR", str(REPO_ROOT / "data"))))
OT_DIR = DATA_DIR / "opentargets"
BUILD_KG_ROOT = DATA_DIR / "kg"
print(f"repo={REPO_ROOT}")
print(f"sample_mode={SAMPLE_MODE} run_build={RUN_BUILD} run_full_validation={RUN_FULL_VALIDATION}")
print(f"default_kg_root={DEFAULT_KG_ROOT}")
print(f"default_kg_root_exists={path_exists(DEFAULT_KG_ROOT)} data_dir={DATA_DIR}")


In [ ]:
from manage_db.audit_edge_evidence import audit_edge_evidence
from manage_db.validate_kg import validate_duckdb
from manage_db.kg_schema import RELATION_BY_NAME
BLOCK1_BROAD_RELATIONS = ["gene_interacts_gene", "pathway_contains_gene", "molecule_targets_gene"]
CANDIDATE_SPLITS = {
    "gene_interacts_gene": ["protein_interacts_protein", "tf_regulates_gene", "tf_binds_enhancer", "transcript_interacts_protein", "transcript_interacts_gene"],
    "pathway_contains_gene": ["pathway_contains_protein"],
    "molecule_targets_gene": ["molecule_targets_protein"],
}
for relation in BLOCK1_BROAD_RELATIONS:
    assert relation in RELATION_BY_NAME, relation
for splits in CANDIDATE_SPLITS.values():
    for relation in splits:
        assert relation in RELATION_BY_NAME, relation

In [ ]:
def parquet_path(kind: str, name: str, root: Path = DEFAULT_KG_ROOT) -> Path:
    return root / kind / f"{name}.parquet"

def parquet_exists(kind: str, name: str, root: Path = DEFAULT_KG_ROOT) -> bool:
    return parquet_path(kind, name, root).exists()

def parquet_rows(path: Path) -> int | None:
    if not path.exists():
        return None
    return pq.ParquetFile(path).metadata.num_rows

def relation_inventory(relations: list[str], root: Path = DEFAULT_KG_ROOT) -> pd.DataFrame:
    rows = []
    for relation in relations:
        edge = parquet_path("edges", relation, root)
        evidence = parquet_path("evidence", relation, root)
        rows.append({
            "relation": relation,
            "edge_file": edge.exists(),
            "edge_rows": parquet_rows(edge),
            "evidence_file": evidence.exists(),
            "evidence_rows": parquet_rows(evidence),
        })
    return pd.DataFrame(rows)

def duckdb_relation_summary(relation: str, root: Path = DEFAULT_KG_ROOT) -> pd.DataFrame:
    evidence_path = parquet_path("evidence", relation, root)
    if not evidence_path.exists():
        return pd.DataFrame([{"relation": relation, "status": "missing evidence parquet", "path": str(evidence_path)}])
    con = duckdb.connect(":memory:")
    try:
        cols = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{evidence_path.as_posix()}')").df()["column_name"].tolist()
        dimensions = [c for c in ["source", "source_dataset", "evidence_type", "predicate", "direction"] if c in cols]
        if not dimensions:
            return pd.DataFrame([{"relation": relation, "status": "no standard provenance columns", "path": str(evidence_path)}])
        select_dims = ", ".join(dimensions)
        query = f"""
            SELECT {select_dims}, count(*) AS rows
            FROM read_parquet('{evidence_path.as_posix()}')
            GROUP BY {select_dims}
            ORDER BY rows DESC
            LIMIT 50
        """
        return con.execute(query).df()
    finally:
        con.close()

def edge_evidence_key_anti_join(relation: str, root: Path = DEFAULT_KG_ROOT) -> pd.DataFrame:
    edge_path = parquet_path("edges", relation, root)
    evidence_path = parquet_path("evidence", relation, root)
    if not edge_path.exists() or not evidence_path.exists():
        return pd.DataFrame([{"relation": relation, "status": "missing edge or evidence parquet", "edge_path": str(edge_path), "evidence_path": str(evidence_path)}])
    con = duckdb.connect(":memory:")
    try:
        query = f"""
        WITH edge_keys AS (
            SELECT relation || '|' || x_id || '|' || y_id AS edge_key
            FROM read_parquet('{edge_path.as_posix()}')
        ), evidence_keys AS (
            SELECT COALESCE(edge_key, relation || '|' || x_id || '|' || y_id) AS edge_key
            FROM read_parquet('{evidence_path.as_posix()}')
        )
        SELECT
          (SELECT count(*) FROM edge_keys) AS edge_rows,
          (SELECT count(*) FROM evidence_keys) AS evidence_rows,
          (SELECT count(*) FROM edge_keys e WHERE NOT EXISTS (SELECT 1 FROM evidence_keys v WHERE v.edge_key = e.edge_key)) AS edges_without_evidence,
          (SELECT count(*) FROM evidence_keys v WHERE NOT EXISTS (SELECT 1 FROM edge_keys e WHERE e.edge_key = v.edge_key)) AS evidence_without_edge
        """
        return con.execute(query).df()
    finally:
        con.close()

def endpoint_type_summary(relation: str, root: Path = DEFAULT_KG_ROOT) -> pd.DataFrame:
    edge_path = parquet_path("edges", relation, root)
    if not edge_path.exists():
        return pd.DataFrame([{"relation": relation, "status": "missing edge parquet", "path": str(edge_path)}])
    con = duckdb.connect(":memory:")
    try:
        return con.execute(f"""
            SELECT x_type, y_type, count(*) AS rows
            FROM read_parquet('{edge_path.as_posix()}')
            GROUP BY 1, 2
            ORDER BY rows DESC
        """).df()
    finally:
        con.close()

## Split/no-split gates

A row can be promoted from a broad relation to a narrower split relation only if: the source assertion is specific enough; endpoint namespaces are native to the split; direction/roles/sign are preserved; evidence is written first or together with the edge; endpoint anti-joins and `audit_edge_evidence` pass in staging. Failing any gate means keep the row in the broad relation and preserve nuance in evidence.

In [ ]:
SPLIT_POLICY = pd.DataFrame([
    {"broad_relation":"gene_interacts_gene", "candidate_split":"protein_interacts_protein", "required_native_assertion":"protein/isoform interaction", "required_endpoint_namespace":"protein/isoform IDs on both ends", "default_decision":"no split unless endpoint and assertion are protein-native"},
    {"broad_relation":"gene_interacts_gene", "candidate_split":"tf_regulates_gene", "required_native_assertion":"TF/regulator controls target gene expression", "required_endpoint_namespace":"regulator gene/gene-product -> target gene", "default_decision":"split only for directed regulation with roles/sign/effect preserved"},
    {"broad_relation":"gene_interacts_gene", "candidate_split":"tf_binds_enhancer", "required_native_assertion":"TF/gene-product binds regulatory interval", "required_endpoint_namespace":"TF gene/gene-product -> enhancer interval", "default_decision":"split only when regulatory interval endpoint and assay/context exist"},
    {"broad_relation":"gene_interacts_gene", "candidate_split":"transcript_interacts_protein", "required_native_assertion":"RNA/transcript binds/interacts with protein", "required_endpoint_namespace":"transcript -> protein/isoform", "default_decision":"split only with transcript and protein endpoints"},
    {"broad_relation":"gene_interacts_gene", "candidate_split":"transcript_interacts_gene", "required_native_assertion":"transcript/RNA regulates or interacts with gene", "required_endpoint_namespace":"transcript -> gene", "default_decision":"split only with transcript endpoint and mechanism"},
    {"broad_relation":"pathway_contains_gene", "candidate_split":"pathway_contains_protein", "required_native_assertion":"protein-native pathway/complex membership", "required_endpoint_namespace":"pathway/complex -> protein/isoform", "default_decision":"do not split GO/Reactome gene membership by projection"},
    {"broad_relation":"molecule_targets_gene", "candidate_split":"molecule_targets_protein", "required_native_assertion":"molecule targets direct protein/isoform product", "required_endpoint_namespace":"molecule -> protein/isoform", "default_decision":"do not split OpenTargets gene target rows by projection"},
])
display(SPLIT_POLICY)

## Current broad relation inventory, subdatabase enumeration, and support audit

In [ ]:
display(relation_inventory(BLOCK1_BROAD_RELATIONS))
for relation in BLOCK1_BROAD_RELATIONS:
    print(f"\n=== {relation}: evidence provenance summary ===")
    display(duckdb_relation_summary(relation))
    print("Endpoint types")
    display(endpoint_type_summary(relation))
present_block1 = [r for r in BLOCK1_BROAD_RELATIONS if parquet_exists("edges", r) or parquet_exists("evidence", r)]
if present_block1:
    try:
        audit = audit_edge_evidence(DEFAULT_KG_ROOT, relations=present_block1)
        display(pd.DataFrame([asdict(report) | {"ok": report.ok} for report in audit.relation_reports.values()]))
    except Exception as exc:
        print(f"audit_edge_evidence failed here: {exc}")
    for relation in present_block1:
        display(edge_evidence_key_anti_join(relation))
else:
    print("No Block 1 cache files present.")

## Candidate split staging skeleton and promotion gates

The actual splitter should live in tested `manage_db` code, not inside this notebook. This notebook sets parameters, calls that code, and audits outputs.

In [ ]:
RUN_SPLIT_BUILD = os.environ.get("JOUVENCE_NOTEBOOK_RUN_BLOCK1_SPLIT", os.environ.get("TXGNN_NOTEBOOK_RUN_BLOCK1_SPLIT", "0")) == "1"
STAGING_KG_ROOT = Path(os.environ.get("JOUVENCE_BLOCK1_STAGING_ROOT", os.environ.get("TXGNN_BLOCK1_STAGING_ROOT", str(REPO_ROOT / "data" / "kg-block1-staging"))))
if RUN_SPLIT_BUILD:
    raise NotImplementedError("Call tested manage_db Block 1 splitter functions here; do not implement row transformation logic inline in the notebook.")
else:
    display(pd.DataFrame([{"broad_relation": broad, "candidate_split": split, "staging_root": str(STAGING_KG_ROOT), "status": "not run; requires tested manage_db splitter + JOUVENCE_NOTEBOOK_RUN_BLOCK1_SPLIT=1"} for broad, splits in CANDIDATE_SPLITS.items() for split in splits]))
if RUN_FULL_VALIDATION and STAGING_KG_ROOT.exists():
    report = validate_duckdb(STAGING_KG_ROOT, threads=2, temp_dir=REPO_ROOT / "artifacts" / "cache" / "duckdb-tmp")
    assert report.total_dangling_edges == 0
else:
    print("Promotion gates not run. Set JOUVENCE_NOTEBOOK_FULL_VALIDATION=1 after producing a staging KG.")